# Titanic — ma’lumotlarni tayyorlash va omon qolishni tahlil qilish

Ushbu notebook topshiriqdagi SQL, Python, yosh bo‘yicha tozalash, Excel eksporti va Power BI dashboard talablarini bajaradi. Dataset: Kaggle Titanic training jadvali, 891 yo‘lovchi va 12 ustun.

## 1. Uchrashuv qaydlari

**Maqsad:** omon qolish sinfga bog‘liqmi, yosh guruhlarida bu farq qanday o‘zgaradi — ko‘rish. Xavfsizlik chiqishlari va jihozlardan yo‘lovchilarning teng foydalanishi masalasiga dalil tayyorlash.

**Vizualizatsiya:** (1) chipta sinfi bo‘yicha omon qolgan va omon qolmaganlar soni uchun guruhlangan ustunli diagramma; (2) sinf bo‘yicha omon qolish foizi; (3) yosh toifasi bo‘yicha omon qolish foizi; (4) sinf × yosh toifasi matritsasi.

**Filtrlar:** yosh toifasi, chipta sinfi va jins. Yosh toifalari: 14 yoshgacha — `Children`; 15–64 — `Youth&Adults`; 65+ — `Seniors`.

**Yakuniy baholash:** datasetdagi sinf omon qolish ulushlari 1-sinfda 62.96%, 2-sinfda 47.28%, 3-sinfda 24.24%. Yuqori sinfda ulush yuqori bo‘lgan. Bu kuzatuv sababiyatni isbotlamaydi; yoshdagi bo‘sh qiymatlarni o‘rtacha bilan to‘ldirish guruh ko‘rsatkichlariga ta’sir qilishi mumkin.


## 2. Datasetni yuklash va SQL namunasi

In [ ]:
import sqlite3
import pandas as pd

DATA_URL = "https://drive.google.com/uc?export=download&id=15cFsnPnHc7KlzV0C9QQ5wG5v8PWqJ65C"
df_raw = pd.read_csv(DATA_URL)
print("O‘lchami:", df_raw.shape)
display(df_raw.head())

In [ ]:
# Asl topshiriqdagi SQL misollarini sqlite3 orqali ishga tushirish.
# DataFrame.to_sql uchun SQLite connection ishlatiladi.
con = sqlite3.connect(":memory:")
df_raw.to_sql("titanic_data", con, if_exists="replace", index=False)
display(pd.read_sql_query("SELECT Name, Sex FROM titanic_data LIMIT 10", con))
display(pd.read_sql_query("SELECT Name, Sex, Age FROM titanic_data LIMIT 10", con))

## 3. Ma’lumotlar tuzilmasi va statistik tavsif

In [ ]:
# Ustun turlari, har bir ustundagi to‘liq qatorlar va xotira sarfi
df_raw.info(memory_usage="deep")
print("Jami xotira: {:,} bayt".format(df_raw.memory_usage(deep=True).sum()))

In [ ]:
# Sonli ustunlarning asosiy statistik ko‘rsatkichlari
display(df_raw.describe().T)
# Matnli ustunlarni ham qo‘shib ko‘rish uchun:
display(df_raw.describe(include="all").T)

## 4. Bo‘sh qiymatlarni aniqlash va Age ustunini tozalash

In [ ]:
missing = df_raw.isna().sum().sort_values(ascending=False)
display(missing.rename("bo‘sh_qiymatlar").to_frame())
print("Age bo‘sh qiymatlari:", int(df_raw["Age"].isna().sum()))
print("Age o‘rtachasi:", df_raw["Age"].mean())

In [ ]:
df = df_raw.copy()
df["Age_original"] = df["Age"]  # boshlang‘ich qiymatlar audit uchun saqlanadi
age_mean = df["Age"].mean()
df["Age"] = df["Age"].fillna(age_mean)
print("Age bo‘sh qolmadimi?", df["Age"].isna().sum() == 0)
df["Age"] = df["Age"].round().astype("int64")
print("Age endi butun sonmi?", pd.api.types.is_integer_dtype(df["Age"]))
print("Age bo‘sh qiymatlari:", int(df["Age"].isna().sum()))
print("Yaxlitlangan yosh statistikasi:")
display(df["Age"].describe().to_frame())

## 5. Yosh guruhlari va yo‘lovchilar soni

In [ ]:
def classify_age(age):
    if age <= 14:
        return "Children"
    elif age <= 64:
        return "Youth&Adults"
    return "Seniors"

df["age_category"] = df["Age"].apply(classify_age)
category_order = ["Children", "Youth&Adults", "Seniors"]
df["age_category"] = pd.Categorical(df["age_category"], categories=category_order, ordered=True)
age_counts = df["age_category"].value_counts(sort=False).rename_axis("age_category").to_frame("passengers")
display(age_counts)

## 6. Omon qolish tahlili

In [ ]:
class_summary = (
    df.groupby("Pclass", as_index=False)
      .agg(passengers=("Survived", "size"), survivors=("Survived", "sum"))
)
class_summary["non_survivors"] = class_summary["passengers"] - class_summary["survivors"]
class_summary["survival_rate"] = class_summary["survivors"] / class_summary["passengers"]
display(class_summary)

In [ ]:
age_summary = (
    df.groupby("age_category", observed=False, as_index=False)
      .agg(passengers=("Survived", "size"), survivors=("Survived", "sum"))
)
age_summary["non_survivors"] = age_summary["passengers"] - age_summary["survivors"]
age_summary["survival_rate"] = age_summary["survivors"] / age_summary["passengers"]
display(age_summary)

# Sinf va yosh guruhlarini birga ko‘rish; 0 ta kuzatuv bo‘lsa ulush NaN qoladi.
class_age = (
    df.groupby(["age_category", "Pclass"], observed=False)
      .agg(passengers=("Survived", "size"), survivors=("Survived", "sum"))
)
class_age["survival_rate"] = class_age["survivors"] / class_age["passengers"].replace(0, pd.NA)
display(class_age.reset_index())

In [ ]:
# So‘ralgan vizualizatsiyalar: omon qolganlar soni, sinf ulushi va yosh guruhi ulushi.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
class_summary.plot(x="Pclass", y=["survivors", "non_survivors"], kind="bar", ax=axes[0], color=["#27877f", "#d6a04c"])
axes[0].set_title("Sinf bo‘yicha yo‘lovchilar soni")
axes[0].set_xlabel("Chipta sinfi")
axes[0].set_ylabel("Yo‘lovchilar")
class_summary.plot(x="Pclass", y="survival_rate", kind="bar", ax=axes[1], color="#4677a8", legend=False)
axes[1].set_title("Omon qolish ulushi: sinf")
axes[1].set_xlabel("Chipta sinfi")
axes[1].set_ylabel("Ulush")
axes[1].set_ylim(0, 1)
age_summary.plot(x="age_category", y="survival_rate", kind="bar", ax=axes[2], color="#b97847", legend=False)
axes[2].set_title("Omon qolish ulushi: yosh guruhi")
axes[2].set_xlabel("Yosh toifasi")
axes[2].set_ylabel("Ulush")
axes[2].set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 7. Excel fayliga eksport

In [ ]:
# Indeks qo‘shilmaydi; Age_original audit uchun, age_category esa Power BI slicer uchun.
output_path = "output.xlsx"
df.to_excel(output_path, index=False)
print(f"Tayyor fayl: {output_path}")

## 8. Power BI dashboard

### Model va DAX

`output.xlsx`dagi `Clean_Data`/`Clean_Data` varag‘ini import qiling. `Pclass`, `Survived`, `Age`, `PassengerId`ni Whole number; `Fare`ni Decimal number; kategoriya va matn maydonlarini Text qilib belgilang.

```DAX
Yo'lovchilar = COUNTROWS('Clean_Data')

Omon qolganlar =
CALCULATE([Yo'lovchilar], 'Clean_Data'[Survived] = 1)

Omon qolmaganlar =
CALCULATE([Yo'lovchilar], 'Clean_Data'[Survived] = 0)

Omon qolish ulushi =
DIVIDE([Omon qolganlar], [Yo'lovchilar], 0)
```

### Sahifa tarkibi

- KPI cards: yo‘lovchilar, omon qolganlar, omon qolish ulushi.
- Clustered column: Pclass bo‘yicha omon qolgan / qolmaganlar soni.
- Bar chart: Pclass bo‘yicha omon qolish ulushi.
- Bar chart: age_category bo‘yicha omon qolish ulushi.
- Matrix: satr `age_category`, ustun `Pclass`, qiymat `Omon qolish ulushi`.
- Slicers: `age_category`, `Pclass`, `Sex`.

Hisoblangan ko‘rsatkichlar: 1-sinf 62.96%; 2-sinf 47.28%; 3-sinf 24.24%. `PowerBI_yoriqnoma.md`da shu sozlamalar alohida berilgan.


## Yakuniy maqsad va baholash

**Maqsad:** sinf va yosh bo‘yicha omon qolish farqlarini ko‘rsatib, favqulodda vaziyatlarda teng foydalanish masalasiga asosli savollar tayyorlash.

**Baholash:** 1-sinfdagi omon qolish ulushi (62.96%) 2-sinfdan (47.28%) va 3-sinfdan (24.24%) yuqori. Yosh bo‘yicha tozalangan guruh natijalari: Children 57.69%, Youth&Adults 36.91%, Seniors 9.09%. Seniors guruhining хэмжээ бага (11 kishi); shu sababli foizni ehtiyotkor talqin qiling. Mean-imputation ham guruh tarkibiga ta’sir qilishi mumkin.
